# Nettoyage et Exploration des Données Météo - Paris 2024

**Projet :** Smart Mobility Paris - Prédiction NO2 sur le périphérique  
**Source :** historique-meteo.net (12 fichiers mensuels, janvier-décembre 2024)  
**Sortie :** `meteo_paris_2024_clean.csv` + figures + statistiques exploratoires

---

### Objectif du notebook

Préparer les données météo journalières de Paris 2024 pour les joindre au pipeline principal (pollution NO2/PM10 + validations IDFM). On va :

1. Charger et concaténer les 12 fichiers mensuels
2. Nettoyer (dates, doublons, valeurs manquantes, outliers)
3. Sélectionner les variables pertinentes pour la pollution
4. Explorer (statistiques + visualisations)
5. Exporter un CSV propre prêt à l'emploi

## 0. Imports et configuration

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
# Chemins relatifs depuis le dossier notebooks/
# Structure du projet :
#   PROJET_ETUDES_M2_PARIS/
#     |-- data/                 <- les 12 fichiers historique-meteo-paris-2024-*.csv
#     |   |-- proceessed/       <- sortie : CSV nettoyé + figures
#     |-- notebooks/            <- ce notebook
INPUT_DIR  = "../data"
OUTPUT_DIR = "../data/proceessed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Chargement des 12 fichiers mensuels

Particularités à gérer :
- **BOM UTF-8** en début de fichier (caractère invisible) → encodage `utf-8-sig`
- **4 lignes de commentaires** au-dessus du vrai header (commençant par `####`) → `skiprows=4`

In [ ]:
pattern = os.path.join(INPUT_DIR, "historique-meteo-paris-2024-*.csv")
files = sorted(glob.glob(pattern))
print(f"Fichiers trouvés : {len(files)}\n")

dfs = []
for f in files:
    df_month = pd.read_csv(
        f,
        skiprows=4,            # ignorer les 4 lignes de commentaires ####
        encoding="utf-8-sig",  # gère le BOM en début de fichier
    )
    dfs.append(df_month)
    print(f"  {os.path.basename(f):45s} -> {len(df_month):3d} jours")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal après concaténation : {len(df)} lignes, {df.shape[1]} colonnes")

In [ ]:
# Aperçu rapide
df.head()

## 2. Nettoyage

### 2.1 Conversion de la date et tri chronologique

In [ ]:
df["DATE"] = pd.to_datetime(df["DATE"], format="%Y-%m-%d")
df = df.sort_values("DATE").reset_index(drop=True)

print(f"Période couverte : {df['DATE'].min().date()} -> {df['DATE'].max().date()}")

### 2.2 Vérification de la complétude

On vérifie qu'on a bien une ligne par jour, sans doublon ni trou.

In [ ]:
n_jours_attendus = (df["DATE"].max() - df["DATE"].min()).days + 1
print(f"Jours attendus : {n_jours_attendus}")
print(f"Jours présents : {df['DATE'].nunique()}")
print(f"Doublons sur la date : {df['DATE'].duplicated().sum()}")

# 2024 est une année bissextile -> 366 jours attendus

### 2.3 Sélection des colonnes pertinentes

Le fichier brut contient **33 colonnes** dont beaucoup sont des profils horaires (TEMPERATURE_9H, WEATHER_CODE_15H...). Pour notre modèle journalier de prédiction NO2, on garde uniquement les **agrégats journaliers** pertinents pour la pollution.

**Justification du choix de chaque variable :**

| Variable | Pourquoi c'est utile pour la pollution |
|---|---|
| `WINDSPEED_MAX_KMH` | Le vent disperse les polluants (effet majeur) |
| `PRECIP_TOTAL_DAY_MM` | La pluie lessive les polluants atmosphériques |
| `PRESSURE_MAX_MB` | Haute pression = stagnation = accumulation des polluants |
| `HUMIDITY_MAX_PERCENT` | Influence la formation de particules secondaires |
| `TEMP_MAX/MIN_C` | Température affecte la chimie atmosphérique |
| `VISIBILITY_AVG_KM` | Proxy direct de la qualité de l'air |
| `CLOUDCOVER_AVG_PERCENT` | Affecte la photochimie |
| `UV_INDEX`, `SUNHOUR` | Activent les réactions photochimiques |

In [ ]:
colonnes_a_garder = [
    "DATE",
    "MAX_TEMPERATURE_C",
    "MIN_TEMPERATURE_C",
    "WINDSPEED_MAX_KMH",
    "PRECIP_TOTAL_DAY_MM",
    "HUMIDITY_MAX_PERCENT",
    "VISIBILITY_AVG_KM",
    "PRESSURE_MAX_MB",
    "CLOUDCOVER_AVG_PERCENT",
    "UV_INDEX",
    "SUNHOUR",
    "TOTAL_SNOW_MM",
]
df_clean = df[colonnes_a_garder].copy()
print(f"Colonnes conservées : {len(colonnes_a_garder)} (sur {df.shape[1]})")

### 2.4 Variables dérivées

Ajout de quelques features utiles pour le modèle :
- `TEMP_AVG_C` : moyenne (max + min) / 2
- `TEMP_RANGE_C` : amplitude thermique journalière
- `MOIS`, `JOUR_SEMAINE`, `WEEKEND` : variables temporelles (le trafic varie selon le jour)

In [ ]:
df_clean["TEMP_AVG_C"]   = (df_clean["MAX_TEMPERATURE_C"] + df_clean["MIN_TEMPERATURE_C"]) / 2
df_clean["TEMP_RANGE_C"] = df_clean["MAX_TEMPERATURE_C"] - df_clean["MIN_TEMPERATURE_C"]
df_clean["MOIS"]         = df_clean["DATE"].dt.month
df_clean["JOUR_SEMAINE"] = df_clean["DATE"].dt.dayofweek  # 0=lundi, 6=dimanche
df_clean["WEEKEND"]      = (df_clean["JOUR_SEMAINE"] >= 5).astype(int)

df_clean.head()

### 2.5 Gestion des valeurs manquantes

In [ ]:
nan_counts = df_clean.isna().sum()
nan_pct    = (nan_counts / len(df_clean) * 100).round(2)
nan_report = pd.DataFrame({"NaN": nan_counts, "Pct (%)": nan_pct})

if nan_counts.sum() > 0:
    print("Colonnes avec valeurs manquantes :")
    print(nan_report[nan_report["NaN"] > 0])
else:
    print("Aucune valeur manquante.")

In [ ]:
# Imputation par interpolation linéaire (s'il y a quelques NaN, ce sera le cas pour
# des séries temporelles continues : la météo de la veille/lendemain est une bonne approximation)
num_cols = df_clean.select_dtypes(include=np.number).columns
df_clean[num_cols] = df_clean[num_cols].interpolate(method="linear", limit_direction="both")

print(f"Valeurs manquantes restantes : {df_clean.isna().sum().sum()}")

### 2.6 Détection des valeurs aberrantes (méthode IQR)

**Important :** on ne supprime PAS les outliers automatiquement. En météo, les valeurs extrêmes correspondent souvent à des événements réels (canicules, tempêtes) qui sont précisément ce qu'on veut capturer dans le modèle. On les **signale** seulement.

In [ ]:
vars_outliers = ["MAX_TEMPERATURE_C", "MIN_TEMPERATURE_C", "WINDSPEED_MAX_KMH",
                 "PRECIP_TOTAL_DAY_MM", "PRESSURE_MAX_MB"]

for col in vars_outliers:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = df_clean[(df_clean[col] < low) | (df_clean[col] > high)]
    print(f"{col:25s} : {len(outliers):3d} outlier(s)  | bornes [{low:.1f}, {high:.1f}]")

## 3. Exploration statistique

### 3.1 Statistiques descriptives

In [ ]:
df_clean.describe().round(2)

### 3.2 Corrélations avec la température moyenne

Sanity check : les variables les plus corrélées avec la température doivent être physiquement cohérentes (UV, ensoleillement..).

In [ ]:
corr_temp = df_clean.corr(numeric_only=True)["TEMP_AVG_C"].drop("TEMP_AVG_C")
corr_temp.sort_values(ascending=False).round(3)

## 4. Visualisations

### 4.1 Évolution annuelle des températures

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_clean["DATE"], df_clean["MAX_TEMPERATURE_C"], label="Max", color="tomato", linewidth=1)
ax.plot(df_clean["DATE"], df_clean["MIN_TEMPERATURE_C"], label="Min", color="steelblue", linewidth=1)
ax.fill_between(df_clean["DATE"], df_clean["MIN_TEMPERATURE_C"], df_clean["MAX_TEMPERATURE_C"],
                alpha=0.2, color="gray")
ax.set_title("Températures journalières à Paris - 2024")
ax.set_xlabel("Date"); ax.set_ylabel("Température (°C)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_temperatures_annuelles.png", dpi=120)
plt.show()

### 4.2 Distributions des variables clés pour la pollution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
vars_pollu = ["WINDSPEED_MAX_KMH", "PRECIP_TOTAL_DAY_MM", "PRESSURE_MAX_MB",
              "HUMIDITY_MAX_PERCENT", "TEMP_AVG_C", "VISIBILITY_AVG_KM"]
for ax, var in zip(axes.flat, vars_pollu):
    ax.hist(df_clean[var].dropna(), bins=30, color="steelblue", edgecolor="white")
    ax.set_title(var)
    ax.set_ylabel("Fréquence")
plt.suptitle("Distributions des variables météo pertinentes pour la pollution", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

### 4.3 Matrice de corrélation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = df_clean.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "Corrélation"}, annot_kws={"size": 8})
ax.set_title("Matrice de corrélation des variables météo")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_correlations.png", dpi=120)
plt.show()

### 4.4 Profils mensuels

In [ ]:
monthly = df_clean.groupby("MOIS").agg({
    "TEMP_AVG_C": "mean",
    "PRECIP_TOTAL_DAY_MM": "sum",
    "WINDSPEED_MAX_KMH": "mean",
    "HUMIDITY_MAX_PERCENT": "mean",
}).round(1)
monthly

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
monthly["TEMP_AVG_C"].plot(kind="bar", ax=axes[0, 0], color="tomato")
axes[0, 0].set_title("Température moyenne (°C)")
monthly["PRECIP_TOTAL_DAY_MM"].plot(kind="bar", ax=axes[0, 1], color="steelblue")
axes[0, 1].set_title("Précipitations totales (mm)")
monthly["WINDSPEED_MAX_KMH"].plot(kind="bar", ax=axes[1, 0], color="seagreen")
axes[1, 0].set_title("Vent max moyen (km/h)")
monthly["HUMIDITY_MAX_PERCENT"].plot(kind="bar", ax=axes[1, 1], color="goldenrod")
axes[1, 1].set_title("Humidité max moyenne (%)")
for ax in axes.flat:
    ax.set_xlabel("Mois")
plt.suptitle("Profils mensuels - Paris 2024", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_profils_mensuels.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Export du fichier nettoyé

Le CSS final est prêt à être joint sur la colonne `DATE` avec :
- les données pollution Airparif (NO2, PM10, PM25)
- les validations IDFM réseau ferré
- les données événements/grèves

In [ ]:
output_csv = f"{OUTPUT_DIR}/meteo_paris_2024_clean.csv"
df_clean.to_csv(output_csv, index=False)

print(f"Fichier sauvegardé : {output_csv}")
print(f"Dimensions : {df_clean.shape[0]} lignes x {df_clean.shape[1]} colonnes")
print(f"\nColonnes finales :")
for c in df_clean.columns:
    print(f"  - {c}")

---

## Bilan

**Synthèse pour le rapport :**

| Étape | Résultat |
|---|---|
| Période couverte | 2024-01-01 à 2024-12-31 (366 jours, année bissextile) |
| Complétude | 100% (aucun jour manquant) |
| Doublons | 0 |
| Colonnes initiales | 33 |
| Colonnes finales | 17 (après sélection + variables dérivées) |
| Variables manquantes | Imputation par interpolation linéaire |
| Outliers | Conservés (événements météo réels) |

**Prochaine étape :** appliquer le même traitement aux données IDFM (validations) pour obtenir une série journalière agrégée, puis joindre tous les datasets sur la date.